# Export Privacy and Digital Consumer Protection Stance Turns

| Item | Description |
|---|---|
| Goal | Export the speech turns underlying the Privacy and digital consumer protection group in the grouped stance figure. |
| Unit and sample | One row per distinct legislator speech turn, identified by `packageId` and `turn_idx`, with an assigned concern or support record in this group. Hearings were held in 1999–2025. |
| Input | The approved topic grouping of T2.1.5 (`topic_groups.parquet`) beside this notebook; saved `track2_assigned.parquet`, `member_turns.parquet`, and `hearings_sst.parquet` from the T2.2 Processed folder on the scratch root. |
| Output | `privacy_and_digital_consumer_protection_stance_turns.csv` in the T2.4 Processed folder on the scratch root. Step 2 prints its full path. |
| Stance fields | `has_concern` and `has_support` indicate whether the turn contains that stance within this group. Both can be true. `n_stance_records` counts the underlying extracted records, not distinct turns. |
| Text fields | `turn_text` is the full cleaned speech turn from the saved upstream input. `concern_text` and `support_text` join the distinct cited passages for each stance, separated by blank lines. |
| Columns | `packageId`, `chamber`, `hearing_year`, `heldDate_first`, `turn_idx`, `bioGuideId`, `legislator`, `turn_text`, `has_concern`, `has_support`, `n_stance_records`, `topic_labels`, `concern_text`, `support_text`, in that order. No other columns are exported. |
| Legislator | Combine honorific and surname, such as `Senator Wyden`. If the surname is missing, retain the honorific alone, such as `The Chairman`. |
| Topics | `topic_labels` contains a JSON list of distinct topic labels rather than numeric identifiers. |
| Interpretation | “Raw” means the existing extracted stances and cleaned speech-turn text. No new model is run and no stance is relabeled. A turn can discuss other groups; stance counts, topic labels, and cited passages cover only this group. |

This notebook runs independently of the figure notebook. It counts every underlying stance record while making the CSV row count match the figure's distinct-turn measure.
| Run | Made 2026-10-22 on the stored T2 outputs. Run again on 2026-10-31 after the inputs moved to the scratch root and the output to the T2.4 Processed folder. No model call. |


## Step 1. Import packages

Use pandas to read the saved data and JSON to preserve multiple topic labels in one CSV row.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

## Step 2. Define file locations

Read the existing source files in place. Save the final data CSV in the `output/T2.4.3/` folder beside this notebook.

In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.4 - Stance by Topic over Time")
NOTEBOOK_NAME = "T2.4.3 - Export Privacy Stance Turns"  # the notebook filename without .ipynb
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
T21_PROCESSED = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries" / "Processed"
T22_PROCESSED = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.2 - Track 2 - Legislator Stance Extraction" / "Processed"
D14_PROCESSED = SCRATCH_FOLDER / "D - Data Collection and Processing/D4 - SST Sample Construction" / "Processed"
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
GROUPS_FILE = T21_PROCESSED / "topic_groups.parquet"
RECORDS_FILE = T22_PROCESSED / "track2_assigned.parquet"
TURNS_FILE = T22_PROCESSED / "member_turns.parquet"
HEARINGS_FILE = D14_PROCESSED / "hearings_sst.parquet"
SAMPLE_FILE = PROCESSED_FOLDER / "privacy_and_digital_consumer_protection_stance_turns.csv"  # written by T2.4.3
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = SAMPLE_FILE
TARGET_GROUP = "Privacy and digital consumer protection"
YEAR_MIN = 1999
YEAR_MAX = 2025
TURN_KEYS = ["packageId", "turn_idx"]
OUTPUT_COLUMNS = [
    "packageId", "chamber", "hearing_year", "heldDate_first", "turn_idx",
    "bioGuideId", "legislator", "turn_text", "has_concern", "has_support",
    "n_stance_records", "topic_labels", "concern_text", "support_text",
]


## Step 3. Select the approved group's topics

Use the mapping directly rather than hard-coding topic identifiers. Stop if a topic belongs to more than one group or the requested group is absent.

In [3]:
topic_groups = pd.read_parquet(GROUPS_FILE)
assert topic_groups["topic_id"].is_unique
assert topic_groups[["group", "topic_id", "label"]].notna().all().all()
selected_topics = topic_groups.loc[topic_groups["group"].eq(TARGET_GROUP)].copy()
assert not selected_topics.empty, "The requested group is absent from the mapping."
print(f"Topic mapping rows: {len(topic_groups):,}; selected topics: {len(selected_topics):,}")
display(selected_topics)

Topic mapping rows: 111; selected topics: 5


,topic_id,label,group
91,65,Consumer privacy and anti-spam legislation,Privacy and digital consumer protection
92,67,Digital identity verification and protection,Privacy and digital consumer protection
93,68,consumer data privacy legislation,Privacy and digital consumer protection
94,92,Children's online privacy and safety legislation,Privacy and digital consumer protection
95,93,Online advertising and consumer choice policies,Privacy and digital consumer protection


## Step 4. Load the group's original stance records

Read the original records for the selected topics. Keep repeated positions within the same turn; they contribute to the stance-record count rather than separate CSV rows.

In [4]:
privacy_records = pd.read_parquet(
    RECORDS_FILE, filters=[("topic_id", "in", selected_topics["topic_id"].tolist())]
)
privacy_records = privacy_records.rename(columns={"sentences": "sentence_ids"})
assert privacy_records["stance"].isin(["concern", "support"]).all()
assert privacy_records["topic_label"].notna().all()
assert privacy_records["topic_label"].isin(selected_topics["label"]).all()
assert privacy_records[TURN_KEYS + ["bioGuideId", "record_idx"]].notna().all().all()
assert not privacy_records.duplicated(TURN_KEYS + ["record_idx"]).any()
print(f"Original stance records in the selected topics: {len(privacy_records):,}")
display(privacy_records[TURN_KEYS + ["record_idx", "topic_id", "stance"]].head(3))

Original stance records in the selected topics: 1,654


,packageId,turn_idx,record_idx,topic_id,stance
0,CHRG-106shrg69984,4,1,68,support
1,CHRG-106shrg69984,4,4,65,concern
2,CHRG-106shrg69984,4,5,65,support


## Step 5. Load hearing dates

Use the hearing held date for the sample window and exported hearing year, matching the figure.

In [5]:
hearings = pd.read_parquet(HEARINGS_FILE, columns=["packageId", "heldDate_first"])
assert hearings["packageId"].is_unique
print(f"Hearing metadata rows: {len(hearings):,}")
display(hearings.head(3))

Hearing metadata rows: 2,352


,packageId,heldDate_first
0,CHRG-104hhrg37345,1996-08-01
1,CHRG-105hhrg38882,1997-02-12
2,CHRG-106hhrg58508,1999-07-13


## Step 6. Attach hearing dates

A many-to-one join attaches one date to every stance record without multiplying observations. Missing dates stop the export.

In [6]:
privacy_records = privacy_records.merge(
    hearings, on="packageId", how="left", validate="many_to_one"
)
assert privacy_records["heldDate_first"].notna().all()
privacy_records["hearing_year"] = pd.to_datetime(privacy_records["heldDate_first"]).dt.year
print(f"Dated stance records: {len(privacy_records):,}")
display(privacy_records[TURN_KEYS + ["heldDate_first", "hearing_year", "stance"]].head(3))

Dated stance records: 1,654


,packageId,turn_idx,heldDate_first,hearing_year,stance
0,CHRG-106shrg69984,4,1999-06-10,1999,support
1,CHRG-106shrg69984,4,1999-06-10,1999,concern
2,CHRG-106shrg69984,4,1999-06-10,1999,support


## Step 7. Apply the figure's sample window

Retain hearings held in 1999–2025. Keep both stances and all original records within the selected group.

In [7]:
records_before_date_filter = len(privacy_records)
privacy_records = privacy_records.loc[
    privacy_records["hearing_year"].between(YEAR_MIN, YEAR_MAX)
].copy()
assert not privacy_records.empty
privacy_records = privacy_records.sort_values(
    ["heldDate_first", "packageId", "turn_idx", "record_idx"]
)
print(f"Stance records in {YEAR_MIN}–{YEAR_MAX}: {len(privacy_records):,}")
print(f"Records outside the window: {records_before_date_filter - len(privacy_records):,}")
display(privacy_records[TURN_KEYS + ["record_idx", "topic_id", "stance"]].head(3))

Stance records in 1999–2025: 1,654
Records outside the window: 0


,packageId,turn_idx,record_idx,topic_id,stance
325,CHRG-106shrg69592,17,1,93,concern
985,CHRG-106shrg69985,4,0,93,support
986,CHRG-106shrg69985,4,1,65,support


## Step 8. Load the original cleaned speech turns

Read turns from the relevant hearings. Preserve the ordered sentence lists so each record's one-based citation can be checked against the original input.

In [8]:
member_turns = pd.read_parquet(
    TURNS_FILE,
    columns=TURN_KEYS + ["bioGuideId", "sentences"],
    filters=[("packageId", "in", privacy_records["packageId"].unique().tolist())],
)
member_turns = member_turns.rename(
    columns={"bioGuideId": "turn_bioGuideId", "sentences": "turn_sentences"}
)
assert not member_turns.duplicated(TURN_KEYS).any()
member_turns["turn_text"] = member_turns["turn_sentences"].str.join(" ")
print(f"Member turns loaded from the selected hearings: {len(member_turns):,}")
display(member_turns[TURN_KEYS + ["turn_bioGuideId"]].head(3))

Member turns loaded from the selected hearings: 65,521


,packageId,turn_idx,turn_bioGuideId
0,CHRG-106shrg69592,1,0000000
1,CHRG-106shrg69592,3,0000000
2,CHRG-106shrg69592,5,0000000


## Step 9. Attach the full speech-turn text

Join on the hearing and turn identifiers. Verify that every stance record finds a turn from the same legislator.

In [9]:
privacy_records = privacy_records.merge(
    member_turns, on=TURN_KEYS, how="left", validate="many_to_one"
)
assert privacy_records["turn_sentences"].notna().all()
assert privacy_records["turn_text"].str.strip().ne("").all()
assert privacy_records["bioGuideId"].eq(privacy_records["turn_bioGuideId"]).all()
print(f"Stance records matched to original text: {len(privacy_records):,}")
display(privacy_records[TURN_KEYS + ["bioGuideId", "stance"]].head(3))

Stance records matched to original text: 1,654


,packageId,turn_idx,bioGuideId,stance
0,CHRG-106shrg69592,17,0000000,concern
1,CHRG-106shrg69985,4,W000779,support
2,CHRG-106shrg69985,4,W000779,support


## Step 10. Recover the cited passages

Sentence identifiers are one-based. Check their bounds and retrieve each record's cited text without changing the stance or topic assignment.

In [10]:
cited_passages = []
sentence_id_lists = []
for sentence_ids, turn_sentences in zip(
    privacy_records["sentence_ids"], privacy_records["turn_sentences"]
):
    sentence_ids = [int(sentence_number) for sentence_number in sentence_ids]
    assert sentence_ids, "A stance record has no cited sentences."
    assert min(sentence_ids) >= 1 and max(sentence_ids) <= len(turn_sentences)
    cited_sentences = [turn_sentences[sentence_number - 1] for sentence_number in sentence_ids]
    cited_passages.append(" ".join(cited_sentences))
    sentence_id_lists.append(sentence_ids)

privacy_records["cited_text"] = cited_passages
privacy_records["sentence_ids"] = sentence_id_lists
assert privacy_records["cited_text"].str.strip().ne("").all()
print(f"Stance records with verified cited text: {len(privacy_records):,}")

Stance records with verified cited text: 1,654


## Step 11. Build one CSV row per speech turn

Export only the requested columns in their specified order. Combine honorific and surname into `legislator`, retaining the honorific alone when the surname is missing. Store distinct topic labels in a JSON list and provide readable concern/support passages. The two stance flags can both be true.

In [11]:
metadata_columns = [
    "packageId", "chamber", "hearing_year", "heldDate_first", "turn_idx",
    "bioGuideId", "surname", "honorific", "turn_text",
]
turn_rows = []
for turn_key, turn_records in privacy_records.groupby(TURN_KEYS, sort=False):
    turn_metadata = turn_records[metadata_columns].drop_duplicates()
    assert len(turn_metadata) == 1, f"Conflicting metadata for turn {turn_key}."
    turn_row = turn_metadata.iloc[0].to_dict()
    concern_records = turn_records.loc[turn_records["stance"].eq("concern")]
    support_records = turn_records.loc[turn_records["stance"].eq("support")]
    turn_row["has_concern"] = not concern_records.empty
    turn_row["has_support"] = not support_records.empty
    turn_row["n_stance_records"] = len(turn_records)
    turn_row["topic_labels"] = json.dumps(
        sorted(turn_records["topic_label"].unique().tolist()), ensure_ascii=False
    )
    turn_row["concern_text"] = "\n\n".join(concern_records["cited_text"].drop_duplicates())
    turn_row["support_text"] = "\n\n".join(support_records["cited_text"].drop_duplicates())
    turn_rows.append(turn_row)

raw_turns = pd.DataFrame(turn_rows)
raw_turns["legislator"] = (
    raw_turns["honorific"].fillna("").str.strip() + " "
    + raw_turns["surname"].fillna("").str.strip()
).str.strip()
missing_surname = raw_turns["surname"].fillna("").str.strip().eq("")
assert raw_turns.loc[missing_surname, "legislator"].eq(
    raw_turns.loc[missing_surname, "honorific"].fillna("").str.strip()
).all()
raw_turns = raw_turns[OUTPUT_COLUMNS]
raw_turns = raw_turns.sort_values(["heldDate_first", "packageId", "turn_idx"])
raw_turns = raw_turns.reset_index(drop=True)
print(f"Unique speech-turn rows: {len(raw_turns):,}")
display(raw_turns[TURN_KEYS + ["has_concern", "has_support", "n_stance_records"]].head(3))

Unique speech-turn rows: 1,086


,packageId,turn_idx,has_concern,has_support,n_stance_records
0,CHRG-106shrg69592,17,True,False,1
1,CHRG-106shrg69985,4,False,True,3
2,CHRG-106shrg69984,1,False,True,2


## Step 12. Check coverage and stance counts

Verify the requested schema, legislator names, and coverage of every record and turn. Compare the stance flags with distinct-turn counts from the original records, including turns expressing both stances.

In [12]:
assert raw_turns.columns.tolist() == OUTPUT_COLUMNS
assert raw_turns["legislator"].str.strip().ne("").all()
assert not raw_turns.duplicated(TURN_KEYS).any()
assert len(raw_turns) == len(privacy_records.drop_duplicates(TURN_KEYS))
assert raw_turns["n_stance_records"].sum() == len(privacy_records)
assert (raw_turns["has_concern"] | raw_turns["has_support"]).all()
expected_concern = privacy_records.loc[privacy_records["stance"].eq("concern")]
expected_support = privacy_records.loc[privacy_records["stance"].eq("support")]
assert raw_turns["has_concern"].sum() == len(expected_concern.drop_duplicates(TURN_KEYS))
assert raw_turns["has_support"].sum() == len(expected_support.drop_duplicates(TURN_KEYS))
both_stance_turns = raw_turns["has_concern"] & raw_turns["has_support"]

print(f"Speech turns: {len(raw_turns):,}")
print(f"Concern turns: {raw_turns['has_concern'].sum():,}")
print(f"Support turns: {raw_turns['has_support'].sum():,}")
print(f"Both stances: {both_stance_turns.sum():,}")
print(f"Underlying stance records counted: {raw_turns['n_stance_records'].sum():,}")

Speech turns: 1,086
Concern turns: 696
Support turns: 476
Both stances: 86
Underlying stance records counted: 1,654


## Step 13. Save the raw-turn CSV

Write UTF-8 CSV with standard quoting for multiline passages and topic-label lists. Reload the file to verify the exact column order and confirm that identifiers, names, counts, text, and topic labels survive the CSV round trip.

In [13]:
raw_turns.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
saved_turns = pd.read_csv(OUTPUT_FILE, keep_default_na=False)
assert saved_turns.columns.tolist() == OUTPUT_COLUMNS
assert len(saved_turns) == len(raw_turns)
assert not saved_turns.duplicated(TURN_KEYS).any()
assert saved_turns["n_stance_records"].sum() == len(privacy_records)
for column in OUTPUT_COLUMNS:
    assert saved_turns[column].astype(str).tolist() == raw_turns[column].astype(str).tolist(), column
for label_text in saved_turns["topic_labels"]:
    labels = json.loads(label_text)
    assert labels and labels == sorted(set(labels))
    assert set(labels).issubset(selected_topics["label"])
print(f"Saved and verified: {OUTPUT_FILE}")
print(f"Rows: {len(saved_turns):,}; columns: {len(saved_turns.columns):,}")

Saved and verified: /scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T - Three Step Text Analysis Pipeline/T2 - Topic Extraction and Stance Analysis/T2.4 - Stance by Topic over Time/Processed/privacy_and_digital_consumer_protection_stance_turns.csv
Rows: 1,086; columns: 14
